# Matrix Product

Reason about optimal multiplication order and cost, then implement a NumPy expression, Einstein-notation contraction, diagonal-matrix products, and a sparse matrix product.


In [1]:
%matplotlib inline
import utils
import numpy as np

# -- start --
## This code will be removed by the course compiler
import shutil

def _generate_inputs(seed):
    rng = np.random.RandomState(seed)
    data = {'task2': [], 'task3': [], 'task4': [], 'task5': []}
    randint_range = 3
    for index in range(10):
        N = rng.randint(2, randint_range)
        M = rng.randint(2, randint_range)
        K = rng.randint(2, randint_range)

        # === task2 ===
        data['task2'].append({
            'A': rng.randint(-randint_range, randint_range, size=[M, N]),
            'B': rng.randint(-randint_range, randint_range, size=[M, K]),
            'C': rng.randint(-randint_range, randint_range, size=[M, K]),
            'x': rng.randint(-randint_range, randint_range, size=[K, 1]),
        })

        # === task3 ===
        data['task3'].append({
            'A': rng.randint(-randint_range, randint_range, size=[N, N]),
            'B': rng.randint(-randint_range, randint_range, size=[N, N]),
            'C': rng.randint(-randint_range, randint_range, size=[N, N]),
            'D': rng.randint(-randint_range, randint_range, size=[N, N]),
        })

        # === task4 ===
        data['task4'].append({
            'A': rng.randint(-randint_range, randint_range, size=[N, N]),
            'D': np.diag(rng.randint(-randint_range, randint_range, size=[N])),
        })

        # === task5 ===
        A = rng.randint(-randint_range, randint_range, size=[N, K])
        num_nonzero = rng.randint(1, K * M + 1)  # L
        rows = rng.randint(0, K, size=num_nonzero)
        cols = rng.randint(0, M, size=num_nonzero)
        vals = rng.randint(-randint_range, randint_range, size=num_nonzero)
        B_sparse = np.vstack([rows, cols, vals])  # shape (3, L)

        data['task5'].append({
            'A': A,
            'B_sparse': B_sparse,
        })
    return data

utils.checker.save_npz(_generate_inputs(0), str(utils.checker.get_data_dir() / 'inputs_public.npz'))
utils.checker.save_npz(_generate_inputs(42), str(utils.checker.get_data_dir() / 'inputs_private.npz'))

if not utils.checker.is_valid_npz('xinputs.npz'):
    shutil.copyfile(str(utils.checker.get_data_dir() / 'inputs_public.npz'), 'xinputs.npz')
# -- end --

inputs_flat = utils.checker.load_answer('xinputs.npz')
inputs = {
    'task2': [
        {key: inputs_flat[f'task2 -> {i} -> {key}'] for key in ('A', 'B', 'C', 'x')}
        for i in range(10)
    ],
    'task3': [
        {key: inputs_flat[f'task3 -> {i} -> {key}'] for key in ('A', 'B', 'C', 'D')}
        for i in range(10)
    ],
    'task4': [
        {key: inputs_flat[f'task4 -> {i} -> {key}'] for key in ('A', 'D')}
        for i in range(10)
    ],
    'task5': [
        {key: inputs_flat[f'task5 -> {i} -> {key}'] for key in ('A', 'B_sparse')}
        for i in range(10)
    ],
}

correct_answer = utils.checker.load_answer(
    'yanswer.npz',
    missing_ok=True,
)

answer = {}


# Matrix Product II

$$A_{5 \times 10} \  B_{10 \times 2}\ C_{2 \times 30}\ D_{30 \times 3}\ E_{3 \times 9}$$

1. What will be the shape of the resulting matrix?
2. How many scalar multiplications are required at best?

In [2]:
# Edit this cell in task.ipynb.
answer['task1'] = {
    '1': [0, 0],
    '2': 0
}

## YOUR CODE HERE
# -- start --
answer['task1'] = {
    '1': [5, 9],
    '2': 180 + 100 + 54 + 90
}
# -- end --


In [3]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'task1'))

OK


# NumPy Expression

Using NumPy, write a function that calculates the 
following expression:

$$\exp(A^T(B + 2C) + 3I) \mathbf x,$$

where $I$ is an identity matrix of the necessary shape.

In [4]:
# Edit this cell in task.ipynb.
def numpy_expression(A, B, C, x):
    res = x
    ## YOUR CODE HERE
    # -- start --
    
    res = np.exp(A.T @ (B + 2 * C) + 3 * np.eye(A.shape[-1], B.shape[-1])) @ x
    # -- end --
    return res


In [5]:
answer['task2'] = []
for one_input in inputs['task2']:
    answer['task2'].append(numpy_expression(**one_input))


In [6]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'task2'))

OK


# Einstein's Rule

In *Tensor Algebra*, a direct generalization of linear algebra to $N$-dimensional arrays called *tensors*, Einstein notation is often used.

It works as follows: if you see a repeated upper and lower index in a formula, that index is summed over.

For example, the following tensor expression, summation and matrix product are equivalent:

$$a_k^l b_l^m = \sum_{l=1}^L a_k^l b_l^m = AB$$

In this notation, a subscript means a row index and a superscript means a column index.

<details>
<summary> Note </summary>

> [!NOTE]
> Also at some point it will be important to know that:
> * lower index represents a covariant dimension of a
> tensor
> * upper index represents a contravariant dimension 
> of tensor. But let us omit this part for now.

</details>

# Task

Calculate the following expression written using Einstein's 
rule:

$$a_k^m b_m^n c_n^o d_l^k$$

In [7]:
# Edit this cell in task.ipynb.
def einsteins_rule(A, B, C, D):
    res = A
    ## YOUR CODE HERE
    # -- start --
    res = D @ A @ B @ C
    # -- end --
    return res


In [8]:
answer['task3'] = []
for one_input in inputs['task3']:
    answer['task3'].append(einsteins_rule(**one_input))


In [9]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'task3'))

OK


# Diagonal Matrix Product

You are given two square matrices: $A$ and $D$, where $A$ is a 
full matrix and $D$ is a diagonal matrix:

$$
A = \begin{bmatrix}
- & \mathbf a_1 & - \\
& \vdots & \\
- & \mathbf a_N & - \\
\end{bmatrix}
$$

$$
D = \textrm{diag}(d_1, d_2, \dots, d_N) = \begin{bmatrix}
d_1 & & & & \\
& d_2 & & & \\
& & d_3 & & \\
& & & \ddots & \\
& & & & d_N 
\end{bmatrix}
$$

Write a program to calculate $DA$ and $AD$ in the fastest possible way.

In [10]:
# Edit this cell in task.ipynb.
def diag_prod_DA(A, D):
    res = A
    ## YOUR CODE HERE
    # -- start --
    res = A * np.diag(D).reshape([-1, 1])
    # -- end --
    return res

def diag_prod_AD(A, D):
    res = A
    ## YOUR CODE HERE
    # -- start --
    res = A * np.diag(D).reshape([1, -1])
    # -- end --
    return res


In [11]:
answer['task4_1'] = []
answer['task4_2'] = []
for one_input in inputs['task4']:
    answer['task4_1'].append(diag_prod_DA(**one_input))
    answer['task4_2'].append(diag_prod_AD(**one_input))


In [12]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'task4_1'))
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'task4_2'))

OK
OK


# Sparse Matrix Product

You are given two matrices of the same shape: $A$ and $B$. Matrix $A$ is full
and is given in the form of `numpy.ndarray`.

The second matrix $B$ is **sparse**. That means that most of its entries are equal to $0$, except for $M$ nonzero entries. This matrix is given
as a set of nonzero elements in the form of a $3 \times M$ `numpy.ndarray` of row-column-value tuples (COO sparse matrix form):

$$
\begin{bmatrix}
r_1 & c_1 & v_1 \\
r_2 & c_2 & v_2 \\
& \vdots & \\
r_M & c_M & v_M \\
\end{bmatrix}
$$

If two items in this structure correspond to the same location, use the latter one.

Write the most efficient program that calculates $AB$.

Also return the ratio between the number of multiplication operations that are needed to calculate the sparse product and the number of operations for the full product.

In [13]:
# Edit this cell in task.ipynb.
def sparse_matrix_product(A, B_sparse):
    res = A
    ratio = 0

    ## YOUR CODE HERE
    # -- start --
    # Shape inference
    N, K = A.shape
    K_b = int(B_sparse[0].max()) + 1
    M = int(B_sparse[1].max()) + 1

    # Output matrix
    C = np.zeros((N, M))

    # Use dictionary to resolve duplicates (only keep latest occurrence)
    index_map = {}
    for r, c, v in B_sparse.T:
        index_map[(int(r), int(c))] = v

    # Sparse multiplication
    for (k, j), v in index_map.items():
        C[:, j] += A[:, k] * v

    # Compute operation ratio
    num_sparse_mults = len(index_map) * N
    num_full_mults = N * K * M
    ratio = num_sparse_mults / num_full_mults

    res = C
    # -- end --

    return res, ratio


In [14]:
answer['task5'] = []
for one_input in inputs['task5']:
    answer['task5'].append(sparse_matrix_product(**one_input))


(3, 2) <- B sparse shape
(3, 1) <- B sparse shape
(3, 1) <- B sparse shape
(3, 1) <- B sparse shape
(3, 3) <- B sparse shape
(3, 2) <- B sparse shape
(3, 3) <- B sparse shape
(3, 1) <- B sparse shape
(3, 3) <- B sparse shape
(3, 3) <- B sparse shape


In [15]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'task5'))

OK


In [16]:
utils.checker.save_npz(answer, str(utils.checker.get_data_dir() / 'answer.npz'))
